In [1]:
#Setup and data loading
import numpy as np
import pandas as pd
import random
import time
import json
from pathlib import Path

In [2]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Load prepared datasets
train_ratings = pd.read_csv(
    PROCESSED_DIR / "train_ratings.csv"
)

test_ratings = pd.read_csv(
    PROCESSED_DIR / "test_ratings.csv"
)

movies = pd.read_csv(
    PROCESSED_DIR / "movies.csv"
)

In [3]:
# Basic validation
print("Train ratings shape:", train_ratings.shape)
print("Test ratings shape:", test_ratings.shape)
print("Movies shape:", movies.shape)

print("\nTrain columns:", train_ratings.columns.tolist())
print("Test columns:", test_ratings.columns.tolist())
print("Movie columns:", movies.columns.tolist())

print("\nMissing values:")
print("Train:", train_ratings.isnull().sum().sum())
print("Test:", test_ratings.isnull().sum().sum())
print("Movies:", movies.isnull().sum().sum())

print("\nDuplicate user-movie pairs:")
print(
    "Train:",
    train_ratings.duplicated(["user_id", "movie_id"]).sum()
)
print(
    "Test:",
    test_ratings.duplicated(["user_id", "movie_id"]).sum()
)

print("\nUnique users:")
print("Train:", train_ratings["user_id"].nunique())
print("Test:", test_ratings["user_id"].nunique())

print("\nData loading and validation complete.")


Train ratings shape: (797758, 4)
Test ratings shape: (202451, 4)
Movies shape: (3883, 3)

Train columns: ['user_id', 'movie_id', 'rating', 'timestamp']
Test columns: ['user_id', 'movie_id', 'rating', 'timestamp']
Movie columns: ['movie_id', 'title', 'genres']

Missing values:
Train: 0
Test: 0
Movies: 0

Duplicate user-movie pairs:
Train: 0
Test: 0

Unique users:
Train: 6040
Test: 6040

Data loading and validation complete.


In [6]:
# Extract all unique genres
movies["genre_list"] = movies["genres"].str.split("|")

all_genres = sorted({
    genre
    for genre_list in movies["genre_list"]
    for genre in genre_list
})

# Create one-hot encoded genre features
genre_features = pd.DataFrame(
    0,
    index=movies.index,
    columns=all_genres
)

for idx, genre_list in movies["genre_list"].items():
    genre_features.loc[idx, genre_list] = 1

# Combine movie IDs with genre features
movie_features = pd.concat(
    [
        movies[["movie_id"]].reset_index(drop=True),
        genre_features.reset_index(drop=True)
    ],
    axis=1
)

print("Number of unique genres:", len(all_genres))
print("Movie feature shape:", movie_features.shape)

print("\nSample movie features:")
print(movie_features.head())

print("\nFeature validation:")
print("Missing values:", movie_features.isnull().sum().sum())
print("Duplicate movie IDs:", movie_features["movie_id"].duplicated().sum())

Number of unique genres: 18
Movie feature shape: (3883, 19)

Sample movie features:
   movie_id  Action  Adventure  Animation  Children's  Comedy  Crime  \
0         1       0          0          1           1       1      0   
1         2       0          1          0           1       0      0   
2         3       0          0          0           0       1      0   
3         4       0          0          0           0       1      0   
4         5       0          0          0           0       1      0   

   Documentary  Drama  Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  \
0            0      0        0          0       0        0        0        0   
1            0      0        1          0       0        0        0        0   
2            0      0        0          0       0        0        0        1   
3            0      1        0          0       0        0        0        0   
4            0      0        0          0       0        0        0        0   

  

In [5]:
# Calculate each user's average rating and rating count
user_features = train_ratings.groupby("user_id").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

# Find each user's preferred genre
train_genres = train_ratings.merge(
    movies[["movie_id", "genre_list"]],
    on="movie_id",
    how="left"
)

# Expand genres for calculating genre preferences
user_genre_ratings = train_genres.explode("genre_list")

genre_preferences = user_genre_ratings.groupby(
    ["user_id", "genre_list"]
)["rating"].mean().reset_index()

# Select the highest-rated genre for each user
preferred_genres = (
    genre_preferences
    .sort_values(
        ["user_id", "rating", "genre_list"],
        ascending=[True, False, True]
    )
    .drop_duplicates("user_id")
    [["user_id", "genre_list"]]
    .rename(columns={"genre_list": "preferred_genre"})
)

# Combine user features
user_features = user_features.merge(
    preferred_genres,
    on="user_id",
    how="left"
)

print("User feature shape:", user_features.shape)

print("\nSample user features:")
print(user_features.head())

print("\nFeature validation:")
print("Missing values:", user_features.isnull().sum().sum())
print("Unique users:", user_features["user_id"].nunique())
print("Unique preferred genres:", user_features["preferred_genre"].nunique())

User feature shape: (6040, 4)

Sample user features:
   user_id  avg_rating  rating_count preferred_genre
0        1    4.190476            42             War
1        2    3.873786           103         Western
2        3    3.975000            40         Western
3        4    4.125000            16           Crime
4        5    3.246835           158       Animation

Feature validation:
Missing values: 0
Unique users: 6040
Unique preferred genres: 18


In [7]:
# Creating interaction dataset

# Add user features to training interactions
interactions = train_ratings.merge(
    user_features,
    on="user_id",
    how="left"
)

# Add movie genre features
interactions = interactions.merge(
    movie_features,
    on="movie_id",
    how="left"
)

# Convert ratings into rewards
REWARD_MAP = {
    1: -1.0,
    2: -0.5,
    3: 0.0,
    4: 0.5,
    5: 1.0
}

interactions["reward"] = interactions["rating"].map(REWARD_MAP)

# Sort interactions by user and timestamp
interactions = interactions.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

print("Interaction dataset shape:", interactions.shape)

print("\nSample interactions:")
print(interactions.head())

print("\nValidation:")
print("Missing values:", interactions.isnull().sum().sum())
print(
    "Duplicate user-movie pairs:",
    interactions.duplicated(["user_id", "movie_id"]).sum()
)
print("Unique users:", interactions["user_id"].nunique())
print("Unique movies:", interactions["movie_id"].nunique())
print("Reward values:", sorted(interactions["reward"].unique()))

Interaction dataset shape: (797758, 26)

Sample interactions:
   user_id  movie_id  rating  timestamp  avg_rating  rating_count  \
0        1      3186       4  978300019    4.190476            42   
1        1      1270       5  978300055    4.190476            42   
2        1      1721       4  978300055    4.190476            42   
3        1      1022       5  978300055    4.190476            42   
4        1      2340       3  978300103    4.190476            42   

  preferred_genre  Action  Adventure  Animation  ...  Film-Noir  Horror  \
0             War       0          0          0  ...          0       0   
1             War       0          0          0  ...          0       0   
2             War       0          0          0  ...          0       0   
3             War       0          0          1  ...          0       0   
4             War       0          0          0  ...          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  reward  
0

In [8]:
#Defining state and action features

# Convert average rating into satisfaction levels
def get_satisfaction(avg_rating):
    if avg_rating < 2.5:
        return "Low"
    elif avg_rating < 3.5:
        return "Medium"
    return "High"


# Add satisfaction to user features
user_features["satisfaction"] = user_features[
    "avg_rating"
].apply(get_satisfaction)

# Define available actions
genre_actions = all_genres

# Map each genre to an action ID
action_to_id = {
    genre: idx
    for idx, genre in enumerate(genre_actions)
}

id_to_action = {
    idx: genre
    for genre, idx in action_to_id.items()
}

# Create state columns
user_features["state"] = list(zip(
    user_features["preferred_genre"],
    user_features["satisfaction"]
))

print("Number of actions:", len(genre_actions))
print("Actions:", genre_actions)

print("\nNumber of unique states:", user_features["state"].nunique())

print("\nSample user states:")
print(
    user_features[
        ["user_id", "preferred_genre", "satisfaction", "state"]
    ].head()
)

print("\nAction mapping:")
print(action_to_id)

Number of actions: 18
Actions: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Number of unique states: 52

Sample user states:
   user_id preferred_genre satisfaction                state
0        1             War         High          (War, High)
1        2         Western         High      (Western, High)
2        3         Western         High      (Western, High)
3        4           Crime         High        (Crime, High)
4        5       Animation       Medium  (Animation, Medium)

Action mapping:
{'Action': 0, 'Adventure': 1, 'Animation': 2, "Children's": 3, 'Comedy': 4, 'Crime': 5, 'Documentary': 6, 'Drama': 7, 'Fantasy': 8, 'Film-Noir': 9, 'Horror': 10, 'Musical': 11, 'Mystery': 12, 'Romance': 13, 'Sci-Fi': 14, 'Thriller': 15, 'War': 16, 'Western': 17}


In [9]:
#Generating transitions
# Start with user-level running statistics
running_stats = user_features.set_index("user_id")[
    ["avg_rating", "rating_count", "preferred_genre"]
].copy()

# Store transitions
transitions = []

# Process each user separately
for user_id, group in interactions.groupby("user_id", sort=False):

    stats = running_stats.loc[user_id].copy()

    # Initial state
    current_state = (
        stats["preferred_genre"],
        get_satisfaction(stats["avg_rating"])
    )

    for _, row in group.iterrows():

        # Identify the movie's genres
        movie_genres = row["genre_list"] if "genre_list" in row else None

        # Get the genre features for this movie
        movie_row = movie_features.loc[
            movie_features["movie_id"] == row["movie_id"]
        ].iloc[0]

        available_genres = [
            genre for genre in genre_actions
            if movie_row[genre] == 1
        ]

        # Use the first listed genre as the action
        action = available_genres[0]

        # Current reward
        reward = row["reward"]

        # Update running average rating
        new_count = stats["rating_count"] + 1
        new_avg = (
            stats["avg_rating"] * stats["rating_count"] + row["rating"]
        ) / new_count

        next_state = (
            stats["preferred_genre"],
            get_satisfaction(new_avg)
        )

        transitions.append({
            "user_id": user_id,
            "state": current_state,
            "action": action_to_id[action],
            "reward": reward,
            "next_state": next_state,
            "done": False
        })

        # Update statistics for the next interaction
        stats["avg_rating"] = new_avg
        stats["rating_count"] = new_count

        current_state = next_state

# Convert to DataFrame
transitions_df = pd.DataFrame(transitions)

print("Transitions shape:", transitions_df.shape)

print("\nSample transitions:")
print(transitions_df.head())

print("\nValidation:")
print("Missing values:", transitions_df.isnull().sum().sum())
print("Unique users:", transitions_df["user_id"].nunique())
print("Unique actions:", transitions_df["action"].nunique())
print("Reward values:", sorted(transitions_df["reward"].unique()))

Transitions shape: (797758, 6)

Sample transitions:
   user_id        state  action  reward   next_state   done
0        1  (War, High)       7     0.5  (War, High)  False
1        1  (War, High)       4     1.0  (War, High)  False
2        1  (War, High)       7     0.5  (War, High)  False
3        1  (War, High)       2     1.0  (War, High)  False
4        1  (War, High)      13     0.0  (War, High)  False

Validation:
Missing values: 0
Unique users: 6040
Unique actions: 18
Reward values: [np.float64(-1.0), np.float64(-0.5), np.float64(0.0), np.float64(0.5), np.float64(1.0)]


In [10]:
#Generating corrected transitions

transitions = []

for user_id, group in interactions.groupby("user_id", sort=False):

    # Start with an empty interaction history
    running_total = 0.0
    running_count = 0

    # Keep the user's preferred genre as their fixed profile feature
    preferred_genre = user_features.loc[
        user_features["user_id"] == user_id,
        "preferred_genre"
    ].iloc[0]

    group = group.reset_index(drop=True)

    for i, row in group.iterrows():

        # Build the current state using past ratings only
        current_avg = (
            running_total / running_count
            if running_count > 0 else 3.0
        )

        current_state = (
            preferred_genre,
            get_satisfaction(current_avg)
        )

        # Identify the movie's genres
        movie_row = movie_features.loc[
            movie_features["movie_id"] == row["movie_id"]
        ].iloc[0]

        available_genres = [
            genre for genre in genre_actions
            if movie_row[genre] == 1
        ]

        # Use the first listed genre as the action
        action = available_genres[0]

        # Reward from the current interaction
        reward = row["reward"]

        # Update history after observing the rating
        running_total += row["rating"]
        running_count += 1

        # Build the next state
        next_avg = running_total / running_count

        next_state = (
            preferred_genre,
            get_satisfaction(next_avg)
        )

        # Mark the final interaction for this user
        done = (i == len(group) - 1)

        transitions.append({
            "user_id": user_id,
            "state": current_state,
            "action": action_to_id[action],
            "reward": reward,
            "next_state": next_state,
            "done": done
        })

transitions_df = pd.DataFrame(transitions)

print("Transitions shape:", transitions_df.shape)

print("\nSample transitions:")
print(transitions_df.head())

print("\nValidation:")
print("Missing values:", transitions_df.isnull().sum().sum())
print("Unique users:", transitions_df["user_id"].nunique())
print("Unique actions:", transitions_df["action"].nunique())
print("Terminal transitions:", transitions_df["done"].sum())
print("Reward values:", sorted(transitions_df["reward"].unique()))

print(
    "\nTransitions per user:",
    transitions_df.groupby("user_id").size().describe()
)

Transitions shape: (797758, 6)

Sample transitions:
   user_id          state  action  reward   next_state   done
0        1  (War, Medium)       7     0.5  (War, High)  False
1        1    (War, High)       4     1.0  (War, High)  False
2        1    (War, High)       7     0.5  (War, High)  False
3        1    (War, High)       2     1.0  (War, High)  False
4        1    (War, High)      13     0.0  (War, High)  False

Validation:
Missing values: 0
Unique users: 6040
Unique actions: 18
Terminal transitions: 6040
Reward values: [np.float64(-1.0), np.float64(-0.5), np.float64(0.0), np.float64(0.5), np.float64(1.0)]

Transitions per user: count    6040.000000
mean      132.079139
std       154.203853
min        16.000000
25%        35.000000
50%        76.000000
75%       166.000000
max      1851.000000
dtype: float64


#### Saving transition dataset

In [12]:
TRANSITIONS_PATH = PROCESSED_DIR / "fitted_q_transitions.csv"

transitions_df.to_csv(
    TRANSITIONS_PATH,
    index=False
)

print("Transitions saved to:", TRANSITIONS_PATH)
print("Saved rows:", len(transitions_df))

# Verify the saved file
saved_transitions = pd.read_csv(TRANSITIONS_PATH)

print("\nReload validation:")
print("Reloaded shape:", saved_transitions.shape)
print("Missing values:", saved_transitions.isnull().sum().sum())
print("Terminal transitions:", saved_transitions["done"].sum())

Transitions saved to: C:\Users\Utsab\Desktop\CineRL\data\processed\fitted_q_transitions.csv
Saved rows: 797758

Reload validation:
Reloaded shape: (797758, 6)
Missing values: 0
Terminal transitions: 6040


##### We'll use scikit-learn's MLPRegressor as the function approximator for Fitted Q-Learning. It will learn the Q-value for a given state-action pair.

In [13]:
import sklearn
from sklearn.neural_network import MLPRegressor

print("Scikit-learn version:", sklearn.__version__)
print("MLPRegressor imported successfully.")

Scikit-learn version: 1.6.1
MLPRegressor imported successfully.


In [14]:
#Preparing state-action features

# Convert state tuples into separate columns
state_features = pd.DataFrame(
    transitions_df["state"].tolist(),
    columns=["preferred_genre", "satisfaction"]
)

# One-hot encode preferred genre
preferred_genre_encoded = pd.get_dummies(
    state_features["preferred_genre"],
    prefix="preferred"
).reindex(
    columns=[f"preferred_{g}" for g in genre_actions],
    fill_value=0
)

# One-hot encode satisfaction
satisfaction_encoded = pd.get_dummies(
    state_features["satisfaction"],
    prefix="satisfaction"
).reindex(
    columns=[
        "satisfaction_Low",
        "satisfaction_Medium",
        "satisfaction_High"
    ],
    fill_value=0
)

# One-hot encode action
action_encoded = np.eye(len(genre_actions))[
    transitions_df["action"].to_numpy()
]

action_encoded = pd.DataFrame(
    action_encoded,
    columns=[f"action_{g}" for g in genre_actions]
)

# Combine all input features
X = pd.concat(
    [
        preferred_genre_encoded.reset_index(drop=True),
        satisfaction_encoded.reset_index(drop=True),
        action_encoded.reset_index(drop=True)
    ],
    axis=1
)

print("Feature matrix shape:", X.shape)
print("\nNumber of input features:", X.shape[1])
print("Missing values:", X.isnull().sum().sum())

print("\nSample features:")
print(X.head())

Feature matrix shape: (797758, 39)

Number of input features: 39
Missing values: 0

Sample features:
   preferred_Action  preferred_Adventure  preferred_Animation  \
0             False                False                False   
1             False                False                False   
2             False                False                False   
3             False                False                False   
4             False                False                False   

   preferred_Children's  preferred_Comedy  preferred_Crime  \
0                 False             False            False   
1                 False             False            False   
2                 False             False            False   
3                 False             False            False   
4                 False             False            False   

   preferred_Documentary  preferred_Drama  preferred_Fantasy  \
0                  False            False              False   
1      

In [15]:
#Prepare next-state action features
# Convert the current feature matrix to float32
X = X.astype(np.float32)

# Collect all unique states from current and next states
all_states = sorted(
    set(transitions_df["state"]) |
    set(transitions_df["next_state"])
)

satisfaction_levels = ["Low", "Medium", "High"]

# Create one row for every state-action combination
candidate_rows = []

for state in all_states:
    preferred_genre, satisfaction = state

    for action_id, action_genre in enumerate(genre_actions):

        row = {}

        # Preferred genre features
        for genre in genre_actions:
            row[f"preferred_{genre}"] = float(
                genre == preferred_genre
            )

        # Satisfaction features
        for level in satisfaction_levels:
            row[f"satisfaction_{level}"] = float(
                level == satisfaction
            )

        # Action features
        for genre in genre_actions:
            row[f"action_{genre}"] = float(
                genre == action_genre
            )

        candidate_rows.append(row)

# Keep exactly the same feature order as X
next_state_action_features = pd.DataFrame(
    candidate_rows,
    columns=X.columns
).astype(np.float32)

print("Unique states:", len(all_states))
print(
    "Next-state action matrix shape:",
    next_state_action_features.shape
)

print("\nExpected action combinations:", len(all_states) * len(genre_actions))
print("Missing values:", next_state_action_features.isnull().sum().sum())
print("Feature columns match:", X.columns.equals(next_state_action_features.columns))
print("Data type:", X.dtypes.unique())

Unique states: 54
Next-state action matrix shape: (972, 39)

Expected action combinations: 972
Missing values: 0
Feature columns match: True
Data type: [dtype('float32')]


In [16]:
#Creating Bellman targets
#Initializing Fitted Q-Learning
# Hyperparameters
GAMMA = 0.90
LEARNING_RATE = 0.001
MAX_ITERATIONS = 10
RANDOM_STATE = 42

# Initial Q-value estimates
initial_q_values = np.zeros(
    len(next_state_action_features),
    dtype=np.float32
)

# Map each state-action combination to its initial Q-value
state_action_q = {
    (state, action_id): 0.0
    for state in all_states
    for action_id in range(len(genre_actions))
}

# Calculate initial Bellman targets
targets = transitions_df["reward"].to_numpy(dtype=np.float32)

print("Fitted Q-Learning initialized.")
print("Gamma:", GAMMA)
print("Maximum iterations:", MAX_ITERATIONS)

print("\nInitial target statistics:")
print("Mean:", targets.mean())
print("Minimum:", targets.min())
print("Maximum:", targets.max())
print("Target shape:", targets.shape)

Fitted Q-Learning initialized.
Gamma: 0.9
Maximum iterations: 10

Initial target statistics:
Mean: 0.30845004
Minimum: -1.0
Maximum: 1.0
Target shape: (797758,)


In [17]:
#Initializing the Fitted Q-Learning model
from sklearn.neural_network import MLPRegressor

# Use a reproducible training sample
TRAIN_SAMPLE_SIZE = 50000

training_sample = transitions_df.sample(
    n=TRAIN_SAMPLE_SIZE,
    random_state=SEED
).copy()

X_train = X.loc[training_sample.index].reset_index(drop=True)

y_train = training_sample["reward"].to_numpy(
    dtype=np.float32
)

# Initialize the neural network
q_model = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
    random_state=SEED,
    batch_size=256
)

print("Model initialized.")
print("Training sample shape:", X_train.shape)
print("Target shape:", y_train.shape)
print("Hidden layers:", q_model.hidden_layer_sizes)
print("Training iterations per fit:", q_model.max_iter)
print("Sample target mean:", round(float(y_train.mean()), 4))

Model initialized.
Training sample shape: (50000, 39)
Target shape: (50000,)
Hidden layers: (32,)
Training iterations per fit: 1
Sample target mean: 0.3089


In [19]:
#Training the fitted Q-Learning model
GAMMA = 0.90
MAX_ITERATIONS = 5
TARGET_CLIP = 1.0

# Map each state to its position in the candidate matrix
state_to_index = {
    state: idx
    for idx, state in enumerate(all_states)
}

# Prepare next-state indices
next_state_indices = training_sample["next_state"].map(
    state_to_index
).to_numpy()

rewards = training_sample["reward"].to_numpy(dtype=np.float32)
dones = training_sample["done"].to_numpy(dtype=bool)

training_history = []

# Initial fit using observed rewards
print("Initializing Q-network with observed rewards...")

q_model.fit(X_train, y_train)

print("Initial model fit complete.")
print("Initial loss:", round(float(q_model.loss_), 5))

# Fitted Q-Learning iterations
for iteration in range(1, MAX_ITERATIONS + 1):

    start_time = time.time()

    # Predict Q-values for all possible state-action pairs
    candidate_q_values = q_model.predict(
        next_state_action_features
    )

    candidate_q_values = candidate_q_values.reshape(
        len(all_states),
        len(genre_actions)
    )

    # Maximum future Q-value
    max_next_q = candidate_q_values.max(axis=1)

    # Calculate Bellman targets
    targets = rewards + GAMMA * (
        1 - dones.astype(np.float32)
    ) * max_next_q[next_state_indices]

    targets = np.clip(targets, -TARGET_CLIP, TARGET_CLIP)

    # Update the network
    q_model.fit(X_train, targets)

    elapsed = time.time() - start_time

    training_history.append({
        "iteration": iteration,
        "loss": q_model.loss_,
        "target_mean": targets.mean(),
        "target_min": targets.min(),
        "target_max": targets.max(),
        "max_q": candidate_q_values.max(),
        "runtime_seconds": elapsed
    })

    print(
        f"Iteration {iteration}/{MAX_ITERATIONS} | "
        f"Loss: {q_model.loss_:.5f} | "
        f"Target mean: {targets.mean():.4f} | "
        f"Target range: [{targets.min():.3f}, {targets.max():.3f}] | "
        f"Max Q: {candidate_q_values.max():.3f} | "
        f"Time: {elapsed:.2f}s"
    )

training_history_df = pd.DataFrame(training_history)

print("\nTraining experiment complete.")
print("Final loss:", round(float(q_model.loss_), 5))

Initializing Q-network with observed rewards...


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Initial model fit complete.
Initial loss: 0.15119


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1/5 | Loss: 0.08795 | Target mean: 0.6513 | Target range: [-1.000, 1.000] | Max Q: 0.747 | Time: 0.28s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


... (2 outputs left out)

Iteration 3/5 | Loss: 0.05329 | Target mean: 0.7930 | Target range: [-1.000, 1.000] | Max Q: 1.064 | Time: 0.26s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 4/5 | Loss: 0.05013 | Target mean: 0.8044 | Target range: [-1.000, 1.000] | Max Q: 1.059 | Time: 0.24s
Iteration 5/5 | Loss: 0.04866 | Target mean: 0.8100 | Target range: [-1.000, 1.000] | Max Q: 1.064 | Time: 0.24s

Training experiment complete.
Final loss: 0.04866


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


In [20]:
#Inspecting unclipped Bellman targets
# Predict current Q-values
candidate_q_values = q_model.predict(
    next_state_action_features
)

candidate_q_values = candidate_q_values.reshape(
    len(all_states),
    len(genre_actions)
)

# Maximum future Q-value for each state
max_next_q = candidate_q_values.max(axis=1)

# Calculate targets WITHOUT clipping
unclipped_targets = rewards + GAMMA * (
    1 - dones.astype(np.float32)
) * max_next_q[next_state_indices]

print("Unclipped Bellman target statistics:")
print("Mean:", round(float(unclipped_targets.mean()), 4))
print("Minimum:", round(float(unclipped_targets.min()), 4))
print("Maximum:", round(float(unclipped_targets.max()), 4))

print("\nQ-value statistics:")
print("Mean:", round(float(candidate_q_values.mean()), 4))
print("Minimum:", round(float(candidate_q_values.min()), 4))
print("Maximum:", round(float(candidate_q_values.max()), 4))

print("\nTraining loss history:")
print(training_history_df[
    ["iteration", "loss", "target_mean", "max_q"]
].to_string(index=False))

Unclipped Bellman target statistics:
Mean: 1.1405
Minimum: -1.0
Maximum: 1.9611

Q-value statistics:
Mean: 0.5477
Minimum: -0.3477
Maximum: 1.0679

Training loss history:
 iteration     loss  target_mean    max_q
         1 0.087952     0.651275 0.746820
         2 0.061237     0.762056 1.036362
         3 0.053287     0.793011 1.064083
         4 0.050129     0.804404 1.058938
         5 0.048660     0.809954 1.063825


In [21]:
#Fitted Q-Learning without target clipping

GAMMA = 0.90
MAX_ITERATIONS = 5

# Create a fresh model
q_model = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
    random_state=SEED,
    batch_size=256
)

training_history = []

# Initialize using observed rewards
print("Initializing model with observed rewards...")

q_model.fit(X_train, y_train)

print("Initial loss:", round(float(q_model.loss_), 5))

for iteration in range(1, MAX_ITERATIONS + 1):

    start_time = time.time()

    # Predict Q-values for all state-action combinations
    candidate_q_values = q_model.predict(
        next_state_action_features
    ).reshape(len(all_states), len(genre_actions))

    # Find the best future Q-value for each state
    max_next_q = candidate_q_values.max(axis=1)

    # Calculate Bellman targets without clipping
    targets = rewards + GAMMA * (
        1 - dones.astype(np.float32)
    ) * max_next_q[next_state_indices]

    # Fit the model
    q_model.fit(X_train, targets)

    elapsed = time.time() - start_time

    training_history.append({
        "iteration": iteration,
        "loss": q_model.loss_,
        "target_mean": targets.mean(),
        "target_min": targets.min(),
        "target_max": targets.max(),
        "max_q": candidate_q_values.max(),
        "runtime_seconds": elapsed
    })

    print(
        f"Iteration {iteration}/{MAX_ITERATIONS} | "
        f"Loss: {q_model.loss_:.5f} | "
        f"Target mean: {targets.mean():.4f} | "
        f"Target range: [{targets.min():.3f}, {targets.max():.3f}] | "
        f"Max Q: {candidate_q_values.max():.3f} | "
        f"Time: {elapsed:.2f}s"
    )

training_history_df = pd.DataFrame(training_history)

print("\nTraining complete.")
print("Final loss:", round(float(q_model.loss_), 5))

Initializing model with observed rewards...


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Initial loss: 0.15119


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1/5 | Loss: 0.15554 | Target mean: 0.8092 | Target range: [-1.000, 1.672] | Max Q: 0.747 | Time: 0.24s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


... (2 outputs left out)

Iteration 3/5 | Loss: 0.16801 | Target mean: 1.6922 | Target range: [-1.000, 2.782] | Max Q: 1.980 | Time: 0.26s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 4/5 | Loss: 0.17652 | Target mean: 2.0672 | Target range: [-1.000, 3.270] | Max Q: 2.522 | Time: 0.25s
Iteration 5/5 | Loss: 0.18540 | Target mean: 2.4013 | Target range: [-1.000, 3.696] | Max Q: 2.996 | Time: 0.25s

Training complete.
Final loss: 0.1854


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


In [22]:
# Stabilizing the training loop
GAMMA = 0.50
MAX_ITERATIONS = 5

training_history = []

# Initial model trained on observed rewards
q_model = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=SEED,
    batch_size=256
)

print("Training initial Q-network...")

q_model.fit(X_train, y_train)

print("Initial loss:", round(float(q_model.loss_), 5))
print("Initial epochs:", q_model.n_iter_)

# Fitted Q-Learning iterations
for iteration in range(1, MAX_ITERATIONS + 1):

    start_time = time.time()

    # Use the previous model to calculate targets
    previous_q_values = q_model.predict(
        next_state_action_features
    ).reshape(len(all_states), len(genre_actions))

    max_next_q = previous_q_values.max(axis=1)

    # Calculate Bellman targets
    targets = rewards + GAMMA * (
        1 - dones.astype(np.float32)
    ) * max_next_q[next_state_indices]

    # Train a new model on the fixed targets
    new_q_model = MLPRegressor(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=30,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=SEED + iteration,
        batch_size=256
    )

    new_q_model.fit(X_train, targets)

    # Replace the previous model only after fitting
    q_model = new_q_model

    elapsed = time.time() - start_time

    training_history.append({
        "iteration": iteration,
        "loss": q_model.loss_,
        "epochs": q_model.n_iter_,
        "target_mean": targets.mean(),
        "target_min": targets.min(),
        "target_max": targets.max(),
        "max_q": previous_q_values.max(),
        "runtime_seconds": elapsed
    })

    print(
        f"Iteration {iteration}/{MAX_ITERATIONS} | "
        f"Loss: {q_model.loss_:.5f} | "
        f"Epochs: {q_model.n_iter_} | "
        f"Target mean: {targets.mean():.4f} | "
        f"Target range: [{targets.min():.3f}, {targets.max():.3f}] | "
        f"Max Q: {previous_q_values.max():.3f} | "
        f"Time: {elapsed:.2f}s"
    )

training_history_df = pd.DataFrame(training_history)

print("\nStabilized training experiment complete.")

Training initial Q-network...
Initial loss: 0.1355
Initial epochs: 17
Iteration 1/5 | Loss: 0.13871 | Epochs: 23 | Target mean: 0.6231 | Target range: [-1.042, 1.421] | Max Q: 0.842 | Time: 4.86s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 2/5 | Loss: 0.13991 | Epochs: 30 | Target mean: 0.7689 | Target range: [-1.000, 1.590] | Max Q: 1.180 | Time: 7.87s
Iteration 3/5 | Loss: 0.14191 | Epochs: 18 | Target mean: 0.8456 | Target range: [-1.000, 1.681] | Max Q: 1.363 | Time: 3.52s


C:\Users\Utsab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 4/5 | Loss: 0.14209 | Epochs: 30 | Target mean: 0.8947 | Target range: [-1.000, 1.781] | Max Q: 1.562 | Time: 9.51s
Iteration 5/5 | Loss: 0.14196 | Epochs: 16 | Target mean: 0.9323 | Target range: [-1.000, 1.814] | Max Q: 1.628 | Time: 3.07s

Stabilized training experiment complete.


In [23]:
#Inspecting the final model
final_q_values = q_model.predict(X_train)

print("Training sample predictions")
print("----------------------------")
print("Mean Q-value:", round(float(final_q_values.mean()), 4))
print("Minimum Q-value:", round(float(final_q_values.min()), 4))
print("Maximum Q-value:", round(float(final_q_values.max()), 4))
print("Standard deviation:", round(float(final_q_values.std()), 4))

print("\nTraining history")
display(training_history_df.round(4))

print("\nTarget growth across iterations")
print(
    training_history_df[
        ["iteration", "target_mean", "target_min", "target_max", "max_q"]
    ].round(4)
)

Training sample predictions
----------------------------
Mean Q-value: 0.939
Minimum Q-value: -0.374
Maximum Q-value: 1.6085
Standard deviation: 0.291

Training history


,iteration,loss,epochs,target_mean,target_min,target_max,max_q,runtime_seconds
0,1,0.1387,23,0.6231,-1.0416,1.4210,0.8420,4.8650
1,2,0.1399,30,0.7689,-1.0000,1.5902,1.1805,7.8679
2,3,0.1419,18,0.8456,-1.0000,1.6813,1.3625,3.5231
3,4,0.1421,30,0.8947,-1.0000,1.7810,1.5619,9.5116
4,5,0.1420,16,0.9323,-1.0000,1.8139,1.6277,3.0746



Target growth across iterations
   iteration  target_mean  target_min  target_max   max_q
0          1       0.6231     -1.0416      1.4210  0.8420
1          2       0.7689     -1.0000      1.5902  1.1805
2          3       0.8456     -1.0000      1.6813  1.3625
3          4       0.8947     -1.0000      1.7810  1.5619
4          5       0.9323     -1.0000      1.8139  1.6277


In [24]:
# Saving the trained model and experiment results
import joblib

MODEL_DIR = PROCESSED_DIR / "fitted_q_learning"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save trained model
joblib.dump(q_model, MODEL_DIR / "fitted_q_model.joblib")

# Save training history
training_history_df.to_csv(
    MODEL_DIR / "fitted_q_training_history.csv",
    index=False
)

# Save configuration
fitted_q_config = {
    "algorithm": "Fitted Q-Learning",
    "model": "MLPRegressor",
    "hidden_layer_sizes": [32],
    "activation": "relu",
    "gamma": GAMMA,
    "learning_rate_init": 0.001,
    "training_sample_size": TRAIN_SAMPLE_SIZE,
    "max_iterations": MAX_ITERATIONS,
    "random_state": SEED,
    "state_features": 21,
    "action_features": 18,
    "total_features": 39,
    "number_of_actions": len(genre_actions),
    "number_of_states": len(all_states)
}

with open(MODEL_DIR / "fitted_q_config.json", "w") as f:
    json.dump(fitted_q_config, f, indent=4)

print("Saved files:")
for file in MODEL_DIR.iterdir():
    print("-", file.name)

Saved files:
- fitted_q_config.json
- fitted_q_model.joblib
- fitted_q_training_history.csv


In [25]:
# Step 3R: Verify Saved Model and Feature Structure

import joblib

loaded_q_model = joblib.load(
    MODEL_DIR / "fitted_q_model.joblib"
)

with open(MODEL_DIR / "fitted_q_config.json", "r") as f:
    loaded_config = json.load(f)

print("Model loaded:", type(loaded_q_model).__name__)
print("Configuration:", loaded_config)

print("\nFeature matrix:")
print("Number of features:", len(X.columns))
print("Feature names:")
print(X.columns.tolist())

print("\nModel prediction test:")
sample_predictions = loaded_q_model.predict(X_train.iloc[:5])
print(np.round(sample_predictions, 4))

Model loaded: MLPRegressor
Configuration: {'algorithm': 'Fitted Q-Learning', 'model': 'MLPRegressor', 'hidden_layer_sizes': [32], 'activation': 'relu', 'gamma': 0.5, 'learning_rate_init': 0.001, 'training_sample_size': 50000, 'max_iterations': 5, 'random_state': 42, 'state_features': 21, 'action_features': 18, 'total_features': 39, 'number_of_actions': 18, 'number_of_states': 54}

Feature matrix:
Number of features: 39
Feature names:
['preferred_Action', 'preferred_Adventure', 'preferred_Animation', "preferred_Children's", 'preferred_Comedy', 'preferred_Crime', 'preferred_Documentary', 'preferred_Drama', 'preferred_Fantasy', 'preferred_Film-Noir', 'preferred_Horror', 'preferred_Musical', 'preferred_Mystery', 'preferred_Romance', 'preferred_Sci-Fi', 'preferred_Thriller', 'preferred_War', 'preferred_Western', 'satisfaction_Low', 'satisfaction_Medium', 'satisfaction_High', 'action_Action', 'action_Adventure', 'action_Animation', "action_Children's", 'action_Comedy', 'action_Crime', 'actio

In [26]:
#Prepare Movie Actions

movie_actions = movies[["movie_id", "title"]].copy()

# Assign the first genre using the training rule
movie_actions["action"] = movies["genre_list"].apply(
    lambda genres: sorted(genres)[0]
)

movie_actions["action_id"] = movie_actions["action"].map(
    action_to_id
)

print("Candidate movies:", len(movie_actions))
print("Movies with valid actions:", movie_actions["action_id"].notna().sum())
print("\nAction distribution:")
print(movie_actions["action"].value_counts())

Candidate movies: 3883
Movies with valid actions: 3883

Action distribution:
action
Drama          1176
Comedy         1024
Action          503
Horror          262
Adventure       155
Crime           131
Documentary     123
Thriller        101
Animation        90
Children's       89
Romance          50
Sci-Fi           46
Mystery          36
Western          33
Film-Noir        25
Musical          25
War              12
Fantasy           2
Name: count, dtype: int64


In [32]:
#Fitted Q-Learning Recommendation Function

def get_fitted_q_recommendations(
    user_id,
    user_features,
    movie_actions,
    model,
    train_ratings,
    k=10
):
    # Get user state
    user_row = user_features[
        user_features["user_id"] == user_id
    ].iloc[0]

    preferred_genre = user_row["preferred_genre"]
    satisfaction = user_row["satisfaction"]

    # Find movies not seen during training
    seen_movies = set(
        train_ratings.loc[
            train_ratings["user_id"] == user_id,
            "movie_id"
        ]
    )

    candidates = movie_actions[
        ~movie_actions["movie_id"].isin(seen_movies)
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    # Build state-action features
    candidate_features = []

    for action in genre_actions:
        row = {}

        for genre in all_genres:
            row[f"preferred_{genre}"] = int(
                preferred_genre == genre
            )

        for level in SATISFACTION_LEVELS:
            row[f"satisfaction_{level}"] = int(
                satisfaction == level
            )

        for genre in all_genres:
            row[f"action_{genre}"] = int(
                action == genre
            )

        candidate_features.append(row)

    state_action_df = pd.DataFrame(
        candidate_features,
        columns=X.columns
    ).astype(np.float32)

    # Predict Q-value for each genre action
    genre_q_values = model.predict(state_action_df)

    q_by_action = dict(
        zip(genre_actions, genre_q_values)
    )

    # Assign each candidate movie its genre Q-value
    candidates["q_value"] = candidates["action"].map(
        q_by_action
    )

    # Rank candidates by predicted Q-value
    recommendations = candidates.sort_values(
        ["q_value", "popularity"],
        ascending=[False, False]
    ).head(k).copy()

    recommendations.insert(
        0, "user_id", user_id
    )

    return recommendations[
        ["user_id", "movie_id", "title", "action", "q_value"]
    ]

In [29]:
# Define satisfaction levels

SATISFACTION_LEVELS = ["Low", "Medium", "High"]

print("Satisfaction levels:", SATISFACTION_LEVELS)

Satisfaction levels: ['Low', 'Medium', 'High']


In [30]:
test_user_id = int(user_features["user_id"].iloc[0])

sample_recommendations = get_fitted_q_recommendations(
    test_user_id,
    user_features,
    movie_actions,
    loaded_q_model,
    train_ratings,
    k=10
)

display(sample_recommendations)

,user_id,movie_id,title,action,q_value
3236,1,3305,Bluebeard (1944),Film-Noir,1.128489
3637,1,3706,Angel Heart (1987),Film-Noir,1.128489
2871,1,2940,Gilda (1946),Film-Noir,1.128489
1232,1,1252,Chinatown (1974),Film-Noir,1.128489
910,1,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Film-Noir,1.128489
736,1,746,Force of Evil (1948),Film-Noir,1.128489
1696,1,1748,Dark City (1998),Film-Noir,1.128489
3311,1,3380,Railroaded! (1947),Film-Noir,1.128489
537,1,541,Blade Runner (1982),Film-Noir,1.128489
901,1,913,"Maltese Falcon, The (1941)",Film-Noir,1.128489


In [31]:
#Calculate Training Movie Popularity

movie_popularity = (
    train_ratings.groupby("movie_id")
    .size()
    .rename("popularity")
    .reset_index()
)

movie_actions = movie_actions.merge(
    movie_popularity,
    on="movie_id",
    how="left"
)

movie_actions["popularity"] = (
    movie_actions["popularity"].fillna(0)
)

print("Movies:", len(movie_actions))
print("Missing popularity values:", movie_actions["popularity"].isna().sum())
print(movie_actions[["movie_id", "title", "action", "popularity"]].head())

Movies: 3883
Missing popularity values: 0
   movie_id                               title     action  popularity
0         1                    Toy Story (1995)  Animation      1831.0
1         2                      Jumanji (1995)  Adventure       568.0
2         3             Grumpier Old Men (1995)     Comedy       342.0
3         4            Waiting to Exhale (1995)     Comedy       110.0
4         5  Father of the Bride Part II (1995)     Comedy       197.0


In [33]:
sample_recommendations = get_fitted_q_recommendations(
    test_user_id,
    user_features,
    movie_actions,
    loaded_q_model,
    train_ratings,
    k=10
)

seen_movies = set(
    train_ratings.loc[
        train_ratings["user_id"] == test_user_id,
        "movie_id"
    ]
)

print("Recommendations:", len(sample_recommendations))
print(
    "Seen movies in recommendations:",
    sample_recommendations["movie_id"].isin(seen_movies).sum()
)

display(sample_recommendations)

Recommendations: 10
Seen movies in recommendations: 0


,user_id,movie_id,title,action,q_value
537,1,541,Blade Runner (1982),Film-Noir,1.128489
1232,1,1252,Chinatown (1974),Film-Noir,1.128489
901,1,913,"Maltese Falcon, The (1941)",Film-Noir,1.128489
1696,1,1748,Dark City (1998),Film-Noir,1.128489
1247,1,1267,"Manchurian Candidate, The (1962)",Film-Noir,1.128489
1264,1,1284,"Big Sleep, The (1946)",Film-Noir,1.128489
2117,1,2186,Strangers on a Train (1951),Film-Noir,1.128489
910,1,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Film-Noir,1.128489
918,1,930,Notorious (1946),Film-Noir,1.128489
3637,1,3706,Angel Heart (1987),Film-Noir,1.128489


In [34]:
#Generating Fitted Q-Learning Recommendation
# Keep only relevant test ratings
test_relevant = test_ratings[
    test_ratings["rating"] >= 4
].copy()

# Remove movies already seen during training
train_seen = train_ratings[
    ["user_id", "movie_id"]
].drop_duplicates()

test_relevant = test_relevant.merge(
    train_seen.assign(seen=True),
    on=["user_id", "movie_id"],
    how="left"
)

test_relevant = test_relevant[
    test_relevant["seen"].isna()
].drop(columns="seen")

# Get evaluation users
evaluation_users = sorted(
    test_relevant["user_id"].unique()
)

print("Evaluation users:", len(evaluation_users))
print("Relevant unseen test ratings:", len(test_relevant))

# Generate recommendations
all_fitted_q_recommendations = []

start_time = time.time()

for i, user_id in enumerate(evaluation_users, start=1):

    recs = get_fitted_q_recommendations(
        user_id,
        user_features,
        movie_actions,
        loaded_q_model,
        train_ratings,
        k=10
    )

    if not recs.empty:
        recs["rank"] = range(1, len(recs) + 1)
        all_fitted_q_recommendations.append(recs)

    if i % 1000 == 0:
        print(f"Processed {i}/{len(evaluation_users)} users")

# Combine results
fitted_q_recommendations = pd.concat(
    all_fitted_q_recommendations,
    ignore_index=True
)

elapsed = time.time() - start_time

print("\nRecommendation generation complete.")
print("Total recommendations:", len(fitted_q_recommendations))
print("Unique users:", fitted_q_recommendations["user_id"].nunique())
print("Runtime:", round(elapsed, 2), "seconds")

# Save recommendations
fitted_q_recommendations.to_csv(
    MODEL_DIR / "fitted_q_recommendations.csv",
    index=False
)

print("Saved fitted_q_recommendations.csv")

Evaluation users: 5975
Relevant unseen test ratings: 106049
Processed 1000/5975 users
Processed 2000/5975 users
Processed 3000/5975 users
Processed 4000/5975 users
Processed 5000/5975 users

Recommendation generation complete.
Total recommendations: 59750
Unique users: 5975
Runtime: 110.24 seconds
Saved fitted_q_recommendations.csv


#### Calculating Evaluation metrics

In [35]:
K = 10

# Relevant test movies for each user
relevant_items = (
    test_relevant.groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

# Recommended movies for each user
recommended_items = (
    fitted_q_recommendations
    .groupby("user_id")["movie_id"]
    .apply(list)
    .to_dict()
)

hits = 0
total_precision = 0
total_recall = 0

for user_id in evaluation_users:

    actual = relevant_items[user_id]
    recommended = recommended_items.get(user_id, [])

    relevant_recommended = len(
        set(recommended) & actual
    )

    if relevant_recommended > 0:
        hits += 1

    total_precision += relevant_recommended / K
    total_recall += relevant_recommended / len(actual)

hit_rate = hits / len(evaluation_users)
precision = total_precision / len(evaluation_users)
recall = total_recall / len(evaluation_users)

catalog_coverage = (
    fitted_q_recommendations["movie_id"].nunique()
    / movies["movie_id"].nunique()
)

fitted_q_metrics = {
    "Model": "Fitted Q-Learning",
    "Hit Rate@10": hit_rate,
    "Precision@10": precision,
    "Recall@10": recall,
    "Catalog Coverage": catalog_coverage
}

fitted_q_metrics_df = pd.DataFrame([fitted_q_metrics])

display(fitted_q_metrics_df.round(4))

,Model,Hit Rate@10,Precision@10,Recall@10,Catalog Coverage
0,Fitted Q-Learning,0.1168,0.0185,0.0118,0.0306


In [36]:
# Save Fitted Q-Learning Evaluation


fitted_q_metrics_df.to_csv(
    MODEL_DIR / "fitted_q_evaluation_metrics.csv",
    index=False
)

# Load previous Q-Learning comparison
previous_metrics = pd.read_csv(
    PROCESSED_DIR / "q_learning_evaluation_comparison.csv"
)

# Combine all three models
final_comparison = pd.concat(
    [previous_metrics, fitted_q_metrics_df],
    ignore_index=True
)

print("Final model comparison:")
display(final_comparison.round(4))

# Save combined comparison
final_comparison.to_csv(
    PROCESSED_DIR / "final_model_comparison.csv",
    index=False
)

print("\nSaved:")
print("- fitted_q_evaluation_metrics.csv")
print("- final_model_comparison.csv")

Final model comparison:


,Unnamed: 0,Q-learning,Popularity baseline,Model,Hit Rate@10,Precision@10,Recall@10,Catalog Coverage
0,Hit Rate@10,0.3006,0.3948,NaN,NaN,NaN,NaN,NaN
1,Precision@10,0.0527,0.0810,NaN,NaN,NaN,NaN,NaN
2,Recall@10,0.0330,0.0477,NaN,NaN,NaN,NaN,NaN
3,Catalog Coverage,0.1028,0.0301,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,Fitted Q-Learning,0.1168,0.0185,0.0118,0.0306



Saved:
- fitted_q_evaluation_metrics.csv
- final_model_comparison.csv


In [37]:
#Standardize and Save Final Comparison

# Load previous results
previous_metrics = pd.read_csv(
    PROCESSED_DIR / "q_learning_evaluation_comparison.csv",
    index_col=0
)

# Convert previous results into one row per model
previous_clean = (
    previous_metrics.T
    .rename_axis("Model")
    .reset_index()
)

# Combine with Fitted Q-Learning
final_comparison = pd.concat(
    [previous_clean, fitted_q_metrics_df],
    ignore_index=True
)

# Keep consistent column order
metric_columns = [
    "Model",
    "Hit Rate@10",
    "Precision@10",
    "Recall@10",
    "Catalog Coverage"
]

final_comparison = final_comparison[metric_columns]

# Save
final_comparison.to_csv(
    PROCESSED_DIR / "final_model_comparison.csv",
    index=False
)

display(final_comparison.round(4))

print("\nSaved final_model_comparison.csv")

,Model,Hit Rate@10,Precision@10,Recall@10,Catalog Coverage
0,Q-learning,0.3006,0.0527,0.0330,0.1028
1,Popularity baseline,0.3948,0.0810,0.0477,0.0301
2,Fitted Q-Learning,0.1168,0.0185,0.0118,0.0306



Saved final_model_comparison.csv
